# Experimenting Securely and Efficiently Using an SSH Bastion Host

[![Basic Bastion Host Topology](https://mermaid.ink/img/pako:eNp1j7EOgkAQRH9ls7X8AIWJgoZaTCw4iw23CtG7I8deYYB_91SiNBabTObNJjMD1k4zpnj11DVwzJUFeB3Apjpwz-Trhv15tiBJ1mNZFiNk1ZZ6aZ2FwvUy8-zH8-rk_O37uSC7v2S_JLhCw95Qq2O74V0LpWHDCtMoNV8o3EWhslOMUhBXPmyNqfjAKwydJuG8pbjLfMzpCZG7Stg?type=png)](https://mermaid.live/edit#pako:eNp1j7EOgkAQRH9ls7X8AIWJgoZaTCw4iw23CtG7I8deYYB_91SiNBabTObNJjMD1k4zpnj11DVwzJUFeB3Apjpwz-Trhv15tiBJ1mNZFiNk1ZZ6aZ2FwvUy8-zH8-rk_O37uSC7v2S_JLhCw95Qq2O74V0LpWHDCtMoNV8o3EWhslOMUhBXPmyNqfjAKwydJuG8pbjLfMzpCZG7Stg "Basic Bastion Host Topology")

This is the most basic setup for deploying a secure cluster of compute nodes. A small VM acts as the bastion host: it is the only instance with a public IP address, and every SSH connection to the bare metal worker nodes is routed through it. This way, jobs can be securely batched to the workers while using only a single public IP address.

This notebook is broken up into 3 parts:

### Spawning the required nodes
1. Create a lease
2. Launch the workers
3. Launch the bastion host and assign it a floating IP
### Testing the connection
1. Ensure that it is possible to reach all experiment nodes via SSH
### Clean up
1. Free all of our resources

The first thing we'll have to do is configure our experiment environment. The bastion host is a VM, so we need a site that offers both VMs and bare metal nodes.

In [ ]:
from chi import context

# Auth is preconfigured on jupyterhub. If using externally,
# you can authenticate with a link instead
from chi.jupyterhub import is_jupyterhub_env
if not is_jupyterhub_env():
    context.use_device_auth()

context.use_site("CHI@TACC")
context.choose_project()

Device authorization enabled.
Now using CHI@TACC:
URL: https://chi.tacc.chameleoncloud.org
Location: Austin, Texas, USA
Support contact: help@chameleoncloud.org


To authenticate please go to: https://auth.chameleoncloud.org/auth/realms/chameleon/device?user_code=UIHR-MHFR


In [ ]:
# jupyterhub provides a keypair. If using externally, select one of your existing keypairs
# and set the path to its private key
if not is_jupyterhub_env():
    import os
    import ipywidgets as widgets
    from chi.server import list_keypair
    keypair_dropdown = widgets.Dropdown(options=[k.name for k in list_keypair()], description="Select Keypair")
    keypair_dropdown.observe(lambda change: context.set("keypair_name", change["new"]), names="value")
    context.set("keypair_name", keypair_dropdown.value)
    display(keypair_dropdown)
    context.set("keypair_private_key", os.path.expanduser("~/.ssh/id_rsa"))

## Create a lease

Since our bastion host is only responsible for facilitating connections to our worker nodes, it needs very little hardware. A small VM is ideal: it leaves every bare metal node free for the experiment itself, and it boots in a minute or two.

We reserve the bare metal worker nodes, a flavor for the bastion VM, and one floating IP for the bastion host. The workers don't need floating IPs, since we'll reach them through the bastion host.

In [ ]:
from chi import lease
from datetime import timedelta
import os

worker_node_type = "compute_cascadelake_r"
# This is for experiments with multiple worker nodes.
# If you only require one node for your experiments,
# you can access the node directly without need for a bastion host.
worker_count = 2
assert worker_count >= 2

my_lease = lease.Lease(f"{os.getenv('USER')}-bastion-host-experiment", duration=timedelta(days=1))
my_lease.add_node_reservation(node_type=worker_node_type, amount=worker_count)
my_lease.add_flavor_reservation(name="m1.small", amount=1)
my_lease.add_fip_reservation(1)
my_lease.submit(idempotent=True)

## Launching the workers

Now, we use our lease to launch the worker nodes. Bare metal nodes take around 10 minutes to boot, so we submit them without waiting, and set up the bastion host while they boot.

In [ ]:
from chi import server

image_name = "CC-Ubuntu24.04"
key_name = None if is_jupyterhub_env() else context.get("keypair_name")

workers = [
    server.Server(
        f"{os.getenv('USER')}-worker-{i}",
        reservation_id=my_lease.node_reservations[0]["id"],
        image_name=image_name,
        key_name=key_name,
    )
    for i in range(worker_count)
]
for worker in workers:
    worker.submit(wait_for_active=False, show=None, idempotent=True)

## Launching the bastion host

The bastion host is a VM using the flavor we reserved. One thing that is critically important is that all your instances are on the same network! Otherwise, you will not be able to route your SSH connections through the bastion host. Here, the bastion host and the workers all use the default network, `sharednet1`.

In [ ]:
bastion = server.Server(
    f"{os.getenv('USER')}-bastion",
    flavor_name=my_lease.get_reserved_flavors()[0].name,
    image_name=image_name,
    key_name=key_name,
)
bastion.submit(idempotent=True)

#### Associating a floating IP address

Next, we'll attach the floating IP address we reserved to our bastion host. We will use this IP address as the entrypoint to all of our workers.

In [ ]:
fip = bastion.associate_floating_ip(my_lease.get_reserved_floating_ips()[0])

#### Security Groups
VMs use security groups. These are unique firewall rules that are configurable via OpenStack and the horizon dashboard. This is a convenient way of configuring your VM's security.

By default, all incoming connections to your VM are blocked. In order to allow remote connections, we'll need to assign the "Allow SSH" security group to our bastion host. We'll find it by listing all the groups below.

In most cases, _this is the ONLY security group you should ever add to your VM._ The bare metal workers don't use security groups, and since they have no floating IP, they can't be reached from the internet.

In [ ]:
from chi import network

existing_groups = network.list_security_groups(name_filter="ssh")
if existing_groups:
    sg = existing_groups.pop()
    print(f"Using existing SSH security group - '{sg.name}'")
else:
    print("Creating new SSH group 'Allow SSH'")
    sg = network.SecurityGroup({"name": "Allow SSH", "description": "Allow incoming SSH connections"})
    sg.add_rule("ingress", "tcp", 22)
    sg.submit()

bastion.add_security_group(sg.id)

Now, ensure we can reach our bastion host.

In [ ]:
bastion.check_connectivity()

## Testing the connection

Now that our bastion host is up, we'll wait for the workers to finish booting, and then make sure that we can reach them via SSH routed through the bastion host.

In [ ]:
for worker in workers:
    worker.wait()

Since our workers have no floating IP, we can't connect to them directly from Jupyter. Instead, we'll connect to their private IPs by routing our SSH connections via the bastion host, which acts as a jump host. For more on SSH jump hosts, read here: https://www.redhat.com/sysadmin/ssh-proxy-bastion-proxyjump

The bastion host only forwards the connection, so your private key never leaves this notebook.

**NOTE**: The servers may have "Active" status, but won't actually be reachable over the network for some time after, so we retry each connection until it succeeds. You can check on the status of your nodes by viewing their console in the Chameleon dashboard.

In [ ]:
import time
from fabric import Connection

worker_connections = []
for worker in workers:
    conn = Connection(
        worker.addresses[worker.network_name][0]["addr"],
        user="cc",
        gateway=bastion.ssh_connection(),
        connect_kwargs={"key_filename": context.get("keypair_private_key")},
    )
    print(f"Attempting connection to {worker.name}...")
    while True:
        try:
            conn.run("echo Hello from $(hostname)!")
            break
        except Exception:
            time.sleep(10)
    worker_connections.append(conn)

Amazing! If the above cell executed, that means you have a _secure_ connection to all your workers via only a single IP address! From here, you can use the `worker_connections` list to batch commands out to your worker nodes. Each one is a Fabric connection, so `put` and `run` upload files and run commands on a worker. Or, you can simply clean up resources in the next step.

## Cleanup
When we're done with our experiment, it's important to clean up all of our resources so that they're available for other researchers who may need them.

### Cleaning up resources

Below are some granular flags which, when set to `True`, will delete applicable resources when the next cell is executed.

**NOTE**: If the lease is deleted, then _all_ resources (servers, IPs, etc.) reserved by it will also be deleted.

In [ ]:
clear_floating_ip = False
clear_workers = False
clear_bastion_server = False

delete_lease_and_free_all_resources = False

In [ ]:
if clear_floating_ip:
    bastion.detach_floating_ip(fip, delete=False)
if clear_workers:
    for worker in workers:
        worker.delete()
if clear_bastion_server:
    bastion.delete(delete_ips=False)
if delete_lease_and_free_all_resources:
    my_lease.delete()